# Replacing the damage subproblem with a trained network

## A PhAST laboratory for Google Colab

This is the third course practical. Compare three damage updates on a
quasi-static three-hole specimen: the classical solve, a learned initial guess
followed by classical correction, and checked direct replacement. All three use
the same mesh, material, loading and mechanics settings.

As in the first practical, define the geometry, mesh, named regions, boundary
conditions, loading and material before running the solver. The configuration
below selects the learned initial-guess route:

```yaml
solver:
  solver_type: quasi_static
  damage_update: learned_proposal                                    # was: classical
  damage_predictor: examples.learned_damage.architectures.mesh_graph_net:create_predictor
  damage_checkpoint: mesh_graph_net.pt
  damage_fallback: true
```

A compatible predictor adapter constructs the model inputs and returns a nodal
damage field. Using another architecture requires a matching adapter, checkpoint
and feature definitions.

## Specimen

A 10 mm square plate with a horizontal edge slot and three circular holes,
pulled vertically under quasi-static loading. The holes and the slot are
geometric voids with traction-free boundaries. Damage is solved only in the surrounding material.

## Learning objectives

On completing this laboratory you should be able to:

1. Identify the damage subproblem inside a staggered scheme and state what a
   predictor replaces.
2. Describe the nodal and edge features a mesh-graph damage model consumes, and
   explain why a wrapper owns that construction.
3. Distinguish a proposal from an audited replacement, and predict what each
   does to the accepted state.
4. Read the admissibility and residual audit, and interpret a fallback.
5. Compare a learned damage field with the finite-element reference.
6. Measure complete cost, including inference and acceptance checks.

## Scope of the comparison

The network was trained on a different specimen, so this exercise examines
transfer to the three-hole geometry. Assess predictions using the classical
reference field, projected residuals and complete runtime on the same hardware.
One checkpoint and specimen provide a limited test of this approach.

<div class="badge-row">
<a class="badge-colab" href="https://colab.research.google.com/github/CEMS-Lab/autumn-school/blob/main/notebooks/study/classroom/03_learning_and_hybrid.ipynb">Open in Colab (published edition)</a> · <a class="badge-link" href="../../notebooks/study/classroom/03_learning_and_hybrid.ipynb">Download notebook</a> · <a class="badge-link" href="../../notebooks/solutions/classroom/03_learning_and_hybrid.ipynb">Download with recap answer</a> · <a class="badge-link" href="../../SETUP.md">Environment setup</a>
</div>


**Day 3 · PhAST practical. Updated 15 September 2026.** Obtain the instructor-supplied mesh_graph_net.pt before starting; Colab requests an upload. Run the notebook to generate the three comparisons and their plots. Fresh whole-notebook timing remains to be recorded.

## 1. Prepare the Colab environment

Use a fresh runtime, as in the first practical. `PHAST_REF` selects the branch
or tag to install, and an empty value selects the default branch. Record the
resolved commit, package version and environment with your results. The
installed solver and the adapter checkout should use the same revision.

This example requires the rational AT2 degradation law, full stress degradation
and the mesh-graph adapter. The supplied code loads the network on the CPU and
uses CPU mechanics. A GPU comparison would require explicitly placing inference
on the GPU and measuring the complete calculation again.


In [ ]:
import importlib.util
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

print("Python:", sys.version.split()[0])
print("Running on Colab:", IN_COLAB)

if IN_COLAB:
    # Gmsh's Python wheel needs libGLU at import time.
    subprocess.run(["apt-get", "-qq", "install", "-y", "libglu1-mesa"], check=True)

PHAST_REPOSITORY = "https://github.com/CEMS-Lab/PhAST.git"
PHAST_REF = ""   # "" selects the default branch; set a tag to pin a revision

requirement = f"phast @ git+{PHAST_REPOSITORY}" + (f"@{PHAST_REF}" if PHAST_REF else "")
if importlib.util.find_spec("phast") is None:
    print("Installing:", requirement)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", requirement], check=True)

import phast

# Importing PhAST selects the non-interactive Agg backend. Restore the inline
# backend so that figures appear in the notebook.
%matplotlib inline

import importlib.metadata
import matplotlib

print("PhAST version:", importlib.metadata.version("phast"))
print("Matplotlib backend:", matplotlib.get_backend())

The predictor adapter ships with PhAST under `examples/learned_damage`, which
pip does not install as an importable package. The cell below puts the
repository checkout on the import path, which is also what the
`damage_predictor` entry in the YAML resolves against.

In [ ]:
import subprocess
import sys
from pathlib import Path

WORK = Path("phast_lab2")
WORK.mkdir(exist_ok=True)
CHECKOUT = WORK / "PhAST"

if not CHECKOUT.exists():
    command = ["git", "clone", "--depth", "1"]
    if PHAST_REF:
        command += ["--branch", PHAST_REF]
    command += [PHAST_REPOSITORY, str(CHECKOUT)]
    subprocess.run(command, check=True, capture_output=True)

if str(CHECKOUT) not in sys.path:
    sys.path.insert(0, str(CHECKOUT))

from examples.learned_damage.architectures import mesh_graph_net

print("Adapter available:", mesh_graph_net.MeshGraphNetPredictor.name)

## 2. Obtain the trained weights

The checkpoint is a TorchScript archive of roughly 850 kB. A TorchScript archive
carries its own architecture, so it evaluates without the model source being
present, and one file is the whole distribution.

Obtain the instructor-supplied checkpoint before starting. The next cell prints
its SHA-256 fingerprint; record it with your results. Verification additionally
requires comparison with the expected fingerprint supplied by the instructor.

In [ ]:
import hashlib
import urllib.request

CHECKPOINT_URL = ""   # release-asset URL, when one is published
CHECKPOINT = WORK / "mesh_graph_net.pt"

if not CHECKPOINT.exists():
    if CHECKPOINT_URL:
        print("Downloading", CHECKPOINT_URL)
        urllib.request.urlretrieve(CHECKPOINT_URL, CHECKPOINT)
    elif IN_COLAB:
        from google.colab import files
        print("Upload mesh_graph_net.pt")
        uploaded = files.upload()
        name = next(iter(uploaded))
        Path(name).replace(CHECKPOINT)
    else:
        raise FileNotFoundError(
            f"Place mesh_graph_net.pt at {CHECKPOINT}, or set CHECKPOINT_URL.")

digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
print("checkpoint:", CHECKPOINT, f"({CHECKPOINT.stat().st_size / 1024:.0f} kB)")
print("sha256:", digest)

model, metadata = mesh_graph_net.load_network(CHECKPOINT, device="cpu")
print("format:", metadata["format"])

## 3. Specimen geometry

Written as a Gmsh script, with the slot and the holes subtracted from the plate
by a boolean operation. Hole positions and radius are parameters, so the same
function serves the exercises.

| Quantity | Value |
|---|---|
| Plate | 10 mm by 10 mm |
| Edge slot | length 2 mm at mid-height, width 0.02 mm |
| Hole centres | (4, 4.4), (6, 5.8), (8, 4.6) mm |
| Hole radius | 0.55 mm |
| Element size | 0.2 mm |
| Phase-field length | 0.4 mm |

The slot and the holes are absent material. Their boundaries are traction-free
surfaces of the domain. Prescribing damage equal to one on a hole boundary would
be a different model, in which the material is present but broken.

In [ ]:
def write_three_hole_geo(path, *, W=10.0, H=10.0, slot_length=2.0,
                         slot_width=0.02,
                         hole_centres=((4.0, 4.4), (6.0, 5.8), (8.0, 4.6)),
                         hole_radius=0.55, h=0.2):
    """Write a Gmsh .geo for a plate with an edge slot and circular holes."""
    centre_y = H / 2.0
    half = slot_width / 2.0
    lines = [
        "// Plate with a horizontal edge slot and circular holes",
        'SetFactory("OpenCASCADE");',
        f"h = {h};",
        f"Rectangle(1) = {{0, 0, 0, {W}, {H}}};",
        f"Rectangle(2) = {{0, {centre_y - half}, 0, {slot_length}, {slot_width}}};",
    ]
    tags = [2]
    for index, (cx, cy) in enumerate(hole_centres, start=3):
        lines.append(f"Disk({index}) = {{{cx}, {cy}, 0, {hole_radius}}};")
        tags.append(index)
    cutters = ", ".join(str(tag) for tag in tags)
    lines.append(
        "BooleanDifference(100) = { Surface{1}; Delete; }"
        f"{{ Surface{{{cutters}}}; Delete; }};")
    lines += [
        "",
        "// Named boundaries recovered by bounding box, so they survive remeshing.",
        "eps = 1e-6;",
        f"bottom() = Curve In BoundingBox{{-eps, -eps, -eps, {W}+eps, eps, eps}};",
        f"top()    = Curve In BoundingBox{{-eps, {H}-eps, -eps, {W}+eps, {H}+eps, eps}};",
        'Physical Curve("bottom") = {bottom()};',
        'Physical Curve("top") = {top()};',
        'Physical Surface("plate") = {100};',
        "",
        "Mesh.CharacteristicLengthMin = h;",
        "Mesh.CharacteristicLengthMax = h;",
        "Mesh.Algorithm = 6;",
        "Mesh.ElementOrder = 1;",
    ]
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path


geo_file = write_three_hole_geo(WORK / "three_hole.geo")
print(geo_file.read_text()[:520])

## 4. Mesh and named regions

Gmsh is driven through its Python API. The mesh is written in the legacy `msh2`
format, which carries the physical group names PhAST reads. The same mesh serves
all three simulations, providing a shared discretisation for the damage-route
comparison.

In [ ]:
import gmsh


def build_mesh(geo_path, msh_path, *, verbose=False):
    """Mesh a .geo file with the Gmsh Python API and write msh2 output."""
    gmsh.initialize()
    try:
        gmsh.option.setNumber("General.Terminal", 1 if verbose else 0)
        gmsh.open(str(geo_path))
        gmsh.model.mesh.generate(2)
        gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
        gmsh.write(str(msh_path))
    finally:
        gmsh.finalize()
    return Path(msh_path)


mesh_file = build_mesh(geo_file, WORK / "three_hole.msh")

mesh = phast.FEMMesh(str(mesh_file), device="cpu")
print(mesh.summary())
print("Named node sets:", sorted(mesh.node_sets.keys()))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.tri import Triangulation

nodes = np.asarray(mesh.nodes.cpu(), dtype=float)
elements = np.asarray(mesh.elements.cpu(), dtype=int)
triangulation = Triangulation(nodes[:, 0], nodes[:, 1], elements)

fig, ax = plt.subplots(figsize=(6.2, 6.0))
ax.triplot(triangulation, lw=0.2, color="0.78")
for name, colour, label in (("bottom", "#336b87", "fixed in x and y"),
                            ("top", "#a94f42", "prescribed u_y, fixed u_x")):
    index = np.asarray(mesh.node_sets[name])
    ax.scatter(nodes[index, 0], nodes[index, 1], s=9, color=colour,
               label=f"{name}: {label}", zorder=3)
ax.set_aspect("equal")
ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_title(f"{mesh.n_nodes} nodes, {mesh.n_elems} T3 elements")
ax.legend(loc="lower center", fontsize=8, framealpha=0.9)
plt.show()

## 5. Material

| Property | Symbol | Value |
|---|---|---|
| Young's modulus | $E$ | 210 000 MPa |
| Poisson's ratio | $\nu$ | 0.30 |
| Critical energy release rate | $G_c$ | 2.7 N/mm |
| Phase-field length scale | $\ell_0$ | 0.4 mm |
| Residual stiffness | $\eta$ | $10^{-7}$ |

Two constitutive choices need stating explicitly, because a frozen network is
only meaningful under the model it was trained on.

**Degradation.** `degradation_type='rational_at2'` selects

$$g(d) = \frac{(1-d)^2}{(1-d)^2 + d} = \frac{(1-d)^2}{d^2 - d + 1}.$$

The law carries no length-scale coefficient and applies no residual-stiffness
blend. It is not quadratic, so the damage subproblem is nonlinear and PhAST
solves it with a bound-constrained projected Newton iteration rather than the
linear conjugate-gradient route used in the first laboratory.

**Stress degradation.** `stress_degradation='full'` degrades the complete
undamaged stress by the single factor $g(d)$, while the history field is still
built from the tensile part of a spectral split. This is a non-variational
hybrid. It is stated here because changing it would change the problem the
frozen network sees.

In [ ]:
material = phast.Material(
    E=210000.0, nu=0.30, Gc=2.7, l0=0.4, rho=7.8e-9,
    eta_residual=1.0e-7,
    energy_split="spectral",
    stress_degradation="full",
    pf_model="AT2",
    degradation_type="rational_at2",
    plane_stress=True,
)

import torch

d = torch.linspace(0.0, 1.0, 6, dtype=torch.float64)
g, gp, _ = material.rational_at2_degradation_derivatives(d)
for value, degraded, slope in zip(d.tolist(), g.tolist(), gp.tolist()):
    print(f"d = {value:.1f}   g(d) = {degraded:.4f}   g'(d) = {slope:+.4f}")
print("\nStress/tangent split under this contract:",
      material.effective_stress_split())

## 6. Boundary conditions, loading and the solver route

The bottom edge is fixed in both components. The top edge is restrained
horizontally and displaced vertically to 0.04 mm over 500 equal increments. The
slot and hole boundaries carry no condition.

This is an **implicit quasi-static** analysis. Inertia is neglected, so the 500
increments describe the loading schedule rather than physical time. Each
loading increment performs one mechanics solve and one damage update
(`max_stagger=1`). This is a single staggered pass; equilibrium of the fully
coupled displacement–damage system is not iterated to convergence within each
increment.

`make_config` builds the configuration and takes the damage route as an
argument, so the three runs below differ only in those entries.

In [ ]:
import yaml

N_STEPS = 500
U_FINAL = 0.04       # mm


def make_config(mesh_path, *, name, damage_update="classical", checkpoint=None,
                predictor=("examples.learned_damage.architectures"
                           ".mesh_graph_net:create_predictor"),
                steps=N_STEPS, u_final=U_FINAL, snapshot_every=5,
                predictor_options=None, **solver_overrides):
    """Return a PhAST configuration for the three-hole specimen."""
    solver = {
        "solver_type": "quasi_static",
        "max_stagger": 1,
        "fail_on_stagger_nonconvergence": False,
        "stagger_tol": 1.0e-8,
        "bounds_method": "post_clamp",
        "damage_tol": 1.0e-6,
        "static_tol": 1.0e-8,
        "damage_max_iter": 5000,
        "static_max_iter": 5000,
        "use_multigrid": False,
        "preconditioner": "jacobi",
        "backend": "auto",
    }
    if damage_update != "classical":
        solver.update({
            "damage_update": damage_update,
            "damage_predictor": predictor,
            "damage_checkpoint": str(checkpoint),
            "damage_predictor_options": dict(predictor_options or
                                             {"representation": "damage"}),
            "damage_fallback": True,
        })
    solver.update(solver_overrides)

    return {
        "schema_version": 1,
        "problem": {"name": name},
        "geometry": {"units": "mm", "mesh_path": str(mesh_path)},
        "material": {
            "E": 210000.0, "nu": 0.30, "Gc": 2.7, "l0": 0.4, "rho": 7.8e-9,
            "eta_residual": 1.0e-7,
            "energy_split": "spectral",
            "stress_degradation": "full",
            "pf_model": "AT2",
            "degradation_type": "rational_at2",
            "plane_stress": True,
        },
        "boundary_conditions": [
            {"nodes": "bottom", "type": "fix", "component": 0},
            {"nodes": "bottom", "type": "fix", "component": 1},
            {"nodes": "top", "type": "fix", "component": 0},
            {"nodes": "top", "type": "prescribe", "component": 1, "value": 1.0},
        ],
        "loading": {
            "protocol": "cyclic",
            "cyclic_phases": f"{u_final}:{steps}",
            "num_steps": steps,
            "dt": 1.0,
        },
        "solver": solver,
        "output": {
            "trajectory": True, "trajectory_format": "zarr",
            "h5_every": snapshot_every, "fast": True, "print_every": 100,
        },
    }


print(yaml.safe_dump(make_config(mesh_file, name="preview")["solver"],
                     sort_keys=False))

The three routes use the same quasi-static mechanics. They differ in how the
damage field is obtained:

| Damage route | What determines the accepted state |
|---|---|
| `classical` | the projected Newton solve of the damage equation |
| `learned_proposal` | the classical solve, started from the network's field |
| `learned_replacement` | the network, if the prediction passes the audit; otherwise the classical solve |

`learned_proposal` uses the predicted field as the initial guess for the
classical damage solve. The classical equation and stopping criteria determine
the accepted result. Compare its field with the reference to assess agreement
at the chosen tolerances. Under
`learned_replacement` the prediction is accepted only after checks on shape,
finiteness, bounds, irreversibility, prescribed values and the projected
residual.

## 7. A helper to run one configuration

Runs are invoked through the public command line interface, and the wall-clock
time of each is recorded so that the cost comparison in Section 12 is measured
rather than assumed.

In [ ]:
import os
import time

RUNS = WORK / "runs"
timings = {}


def run_case(tag, **config_kwargs):
    """Write a configuration, run it, and return the output directory."""
    config = make_config(mesh_file, name=tag, **config_kwargs)
    config_file = WORK / f"{tag}.yaml"
    config_file.write_text(yaml.safe_dump(config, sort_keys=False),
                           encoding="utf-8")
    run_dir = RUNS / tag

    environment = dict(os.environ)
    environment["PYTHONPATH"] = (
        str(CHECKOUT) + os.pathsep + environment.get("PYTHONPATH", ""))

    start = time.perf_counter()
    outcome = subprocess.run(
        [sys.executable, "-m", "phast", "run", str(config_file),
         "--device", "cpu", "--output_dir", str(run_dir)],
        capture_output=True, text=True, env=environment)
    elapsed = time.perf_counter() - start
    if outcome.returncode != 0:
        print(outcome.stdout[-3000:])
        print(outcome.stderr[-3000:])
        raise RuntimeError(f"Run {tag} failed")

    timings[tag] = elapsed
    for line in outcome.stdout.splitlines():
        if "predictor=" in line or "damage_update=" in line:
            print(line.strip())
    print(f"{tag}: {elapsed:.1f} s")
    return run_dir

## 8. Run the finite-element reference

Run the classical calculation to obtain the reference field and measured runtime
for the two learned routes. Whole-notebook Colab timing remains to be recorded
for this edition, including all three solves, comparisons and plotting.


In [ ]:
classical_dir = run_case("classical", damage_update="classical")

## 9. Post-processing toolkit

The same reader as the first laboratory, with one change: the loading parameter
is applied displacement rather than physical time, so it is derived from the
stored step index and the prescribed schedule.

In [ ]:
import zarr


class RunViewer:
    """Read-only view of a finished quasi-static PhAST run."""

    _RAW_KEYS = ("step", "displacement", "stress", "strain",
                 "damage_nodal", "H_elem", "H_nodal")

    def __init__(self, run_dir, *, steps=N_STEPS, u_final=U_FINAL):
        self.run_dir = Path(run_dir)
        self.result = phast.load_result(self.run_dir)
        self.mesh = phast.FEMMesh(str(self.run_dir / "mesh.msh"), device="cpu")
        self.nodes = np.asarray(self.mesh.nodes.cpu(), dtype=float)
        self.elements = np.asarray(self.mesh.elements.cpu(), dtype=int)
        self.triangulation = Triangulation(
            self.nodes[:, 0], self.nodes[:, 1], self.elements)
        self._raw = self._load_trajectory()
        self.steps = self._raw["step"]
        # The prescribed schedule is linear, so applied displacement follows
        # directly from the increment index.
        self.applied = (self.steps + 1) / steps * u_final

    def _load_trajectory(self):
        root = zarr.open(str(self.run_dir / "training_data.zarr"), mode="r")
        trajectory = root["simulation_data"]["trajectory"]
        count = int(trajectory.attrs.get("count", len(trajectory["step"])))
        raw = {key: np.asarray(trajectory[key][:count])
               for key in self._RAW_KEYS if key in trajectory}
        raw["step"] = raw["step"].astype(int)
        return raw

    def damage(self, index=-1):
        return self._raw["damage_nodal"][index].astype(float)

    def history_element(self, index):
        return self._raw["H_elem"][index].astype(float)

    def displacement(self, index):
        return self._raw["displacement"][index].astype(float)

    def plot_damage(self, index=-1, *, ax=None, title=None, vmin=0.0, vmax=1.0):
        values = self.damage(index)
        if ax is None:
            _, ax = plt.subplots(figsize=(5.2, 5.0))
        art = ax.tripcolor(self.triangulation, values, shading="gouraud",
                           cmap="inferno", vmin=vmin, vmax=vmax)
        ax.set_aspect("equal")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(title or
                     f"Damage, u = {self.applied[index]:.4f} mm")
        ax.figure.colorbar(art, ax=ax, shrink=0.84)
        return ax


def relative_l2(candidate, reference):
    """Relative nodal L2 difference between two damage fields, in per cent."""
    denominator = np.linalg.norm(reference)
    if denominator == 0.0:
        return float("nan")
    return 100.0 * np.linalg.norm(candidate - reference) / denominator


reference = RunViewer(classical_dir)
print(f"{len(reference.steps)} stored increments, "
      f"applied displacement up to {reference.applied[-1]:.4f} mm")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
for ax, frac in zip(axes, (0.5, 0.75, 1.0)):
    index = int(frac * (len(reference.steps) - 1))
    reference.plot_damage(index, ax=ax)
fig.suptitle("Finite-element reference: damage development", fontsize=12)
fig.tight_layout()
plt.show()

## 10. What the predictor receives, and what it returns

PhAST calls the predictor once per damage subproblem and hands it a
`DamageStepContext` describing the current finite-element state: node
coordinates, element connectivity, displacement, velocity, the element and nodal
history fields, the previously accepted damage, material data, and the step
index. The context also carries architecture-neutral finite-element helpers, so
a wrapper contains only model-specific code:

| Helper | Returns |
|---|---|
| `canonical_node_features()` | `[x, y, H, d_prev, u_x, u_y]` per node |
| `graph_edge_index()` | directed, duplicate-free mesh edges |
| `element_areas()` | area of every triangle |
| `assemble_element_to_nodes(v)` | $\sum_e \int N_i\, v_e\, \mathrm{d}\Omega$ |
| `boundary_node_mask()` | topological boundary nodes, interior voids included |
| `edge_lengths(edge_index)` | length of every edge |

This network consumes two nodal features and one edge feature:

$$
\text{node}_0 = \sum_e \int N_i \frac{\mathcal{H}_e}{G_c \ell_0}\,\mathrm{d}\Omega,
\qquad
\text{node}_1 = \text{boundary indicator},
\qquad
\text{edge}_0 = \frac{|\mathbf{x}_i - \mathbf{x}_j|}{\ell_0}.
$$

The network applies $\log_{10}(1+x)$ to the first column internally.
Reproducing the training-time construction exactly is the wrapper's
responsibility. A mismatch in normalisation or feature order still produces a
plausible crack while no longer representing the trained operator.

The predictor returns one value per node. PhAST uses this field according to
the selected damage route and its acceptance checks.

In [ ]:
from phast.learned_damage import DamageStepContext

predictor = mesh_graph_net.create_predictor(
    checkpoint=CHECKPOINT, device="cpu", options={"representation": "damage"})


def context_at(view, index):
    """Rebuild the damage-step context from a stored increment."""
    n_nodes = view.nodes.shape[0]
    previous = max(index - 1, 0)
    return DamageStepContext(
        step=int(view.steps[index]), time=None, load_factor=1.0,
        nodes=view.mesh.nodes, elements=view.mesh.elements,
        displacement=torch.as_tensor(view.displacement(index),
                                     dtype=torch.float64),
        velocity=torch.zeros((n_nodes, 2), dtype=torch.float64),
        history_element=torch.as_tensor(view.history_element(index),
                                        dtype=torch.float64).reshape(-1),
        history_nodal=torch.zeros(n_nodes, dtype=torch.float64),
        damage_previous=torch.as_tensor(view.damage(previous),
                                        dtype=torch.float64),
        material={"Gc": 2.7, "l0": 0.4, "E": 210000.0, "nu": 0.30},
        phase_field_model="AT2", energy_split="spectral",
        device=torch.device("cpu"), dtype=torch.float64,
    )


sample = context_at(reference, len(reference.steps) - 1)
features = predictor.node_features(sample)
edge_index = sample.graph_edge_index()

print("nodes:", features.shape[0], " edges:", edge_index.shape[1])
print("node feature 0 (assembled H):",
      f"[{float(features[:, 0].min()):.3g}, {float(features[:, 0].max()):.3g}]")
print("node feature 1 (boundary):",
      sorted(set(features[:, 1].tolist())))
print("edge feature 0 (length / l0):",
      f"[{float(sample.edge_lengths(edge_index).min() / 0.4):.3f}, "
      f"{float(sample.edge_lengths(edge_index).max() / 0.4):.3f}]")

Before putting the network inside the solver loop, ask it to predict the damage
field at stored increments of the finite-element run and compare directly. This
separates the quality of the model from the behaviour of the coupling.

In [ ]:
sampled = range(0, len(reference.steps), 4)
errors, applied, out_of_bounds = [], [], []

for index in sampled:
    predicted = predictor.predict(context_at(reference, index)).damage.numpy()
    truth = reference.damage(index)
    errors.append(relative_l2(predicted, truth))
    applied.append(reference.applied[index])
    out_of_bounds.append(float(max(predicted.max() - 1.0, -predicted.min(), 0.0)))

fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.2))
axes[0].plot(applied, errors, color="#a94f42")
axes[0].set_xlabel("Applied displacement [mm]")
axes[0].set_ylabel("Relative nodal L2 error [%]")
axes[0].set_title("Prediction against the finite-element reference")
axes[0].grid(alpha=0.25)

axes[1].plot(applied, out_of_bounds, color="#336b87")
axes[1].set_xlabel("Applied displacement [mm]")
axes[1].set_ylabel("Largest excursion outside [0, 1]")
axes[1].set_title("Bound violation of the raw prediction")
axes[1].grid(alpha=0.25)

fig.tight_layout()
plt.show()

half = len(errors) // 2
print(f"Relative L2 error over the second half of the loading: "
      f"{np.nanmin(errors[half:]):.1f} to {np.nanmax(errors[half:]):.1f} %")

In [ ]:
index = len(reference.steps) - 1
predicted = predictor.predict(context_at(reference, index)).damage.numpy()
truth = reference.damage(index)

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.3))
reference.plot_damage(index, ax=axes[0], title="Finite-element damage")

art = axes[1].tripcolor(reference.triangulation, predicted, shading="gouraud",
                        cmap="inferno", vmin=0.0, vmax=1.0)
axes[1].set_title("Network prediction")
fig.colorbar(art, ax=axes[1], shrink=0.84)

difference = predicted - truth
bound = float(np.abs(difference).max())
art = axes[2].tripcolor(reference.triangulation, difference, shading="gouraud",
                        cmap="RdBu_r", vmin=-bound, vmax=bound)
axes[2].set_title(f"Difference, relative L2 = {relative_l2(predicted, truth):.1f} %")
fig.colorbar(art, ax=axes[2], shrink=0.84)

for ax in axes[1:]:
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()

## 11. Put the predictor inside the solver loop

Two runs, each differing from Section 8 by the damage route alone.

`learned_proposal` projects the prediction and uses it as the starting point for
the classical solve. Assess agreement with the reference at the chosen solver
tolerances, together with the effect on runtime.

`learned_replacement` accepts the prediction directly, and only after the audit
passes. With fallback enabled, a rejected prediction returns to the classical
solve for that increment.

In [ ]:
proposal_dir = run_case("learned_proposal",
                        damage_update="learned_proposal",
                        checkpoint=CHECKPOINT)

In [ ]:
replacement_dir = run_case("learned_replacement",
                           damage_update="learned_replacement",
                           checkpoint=CHECKPOINT)

## 12. Compare the three runs

Read the damage fields, the trajectory of the difference, and the cost together.

In [ ]:
proposal = RunViewer(proposal_dir)
replacement = RunViewer(replacement_dir)

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.4))
reference.plot_damage(-1, ax=axes[0], title="classical")
proposal.plot_damage(-1, ax=axes[1], title="learned_proposal")
replacement.plot_damage(-1, ax=axes[2], title="learned_replacement")
fig.suptitle("Final damage field, identical specimen and loading", fontsize=12)
fig.tight_layout()
plt.show()

for name, view in (("learned_proposal", proposal),
                   ("learned_replacement", replacement)):
    print(f"{name:22s} final relative L2 difference from classical: "
          f"{relative_l2(view.damage(-1), reference.damage(-1)):.4f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
for name, view, colour in (("learned_proposal", proposal, "#336b87"),
                           ("learned_replacement", replacement, "#a94f42")):
    trajectory = [relative_l2(view.damage(i), reference.damage(i))
                  for i in range(len(reference.steps))]
    ax.plot(reference.applied, trajectory, label=name, color=colour)
ax.set_xlabel("Applied displacement [mm]")
ax.set_ylabel("Relative L2 difference from classical [%]")
ax.set_title("Accepted damage state against the finite-element reference")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

The cost comparison must include everything the learned route actually does:
inference, the projection, the admissibility and residual audit, any rejection,
and the classical fallback that follows a rejection. Compare complete runs on the same hardware and report inference time as one
component of that total.

In [ ]:
baseline = timings["classical"]
fig, ax = plt.subplots(figsize=(6.4, 3.6))
names = list(timings)
values = [timings[name] for name in names]
ax.barh(names, values, color=["#7a8288", "#336b87", "#a94f42"])
for index, value in enumerate(values):
    ax.text(value, index, f"  {value:.1f} s  ({value / baseline:.2f}x)",
            va="center", fontsize=9)
ax.set_xlabel("Wall-clock time [s]")
ax.set_xlim(0, max(values) * 1.35)
ax.set_title("Measured end-to-end cost on this runtime")
plt.show()

print("These timings describe this Colab machine only. No speed-up is claimed.")

## 13. Audit the accepted damage field

A damage field is admissible when it satisfies the box-constrained
Karush-Kuhn-Tucker conditions of the bound-constrained damage problem: the
irreversibility interval $d_{n-1} \le d \le 1$ is respected, the projected
stationarity residual vanishes on interior nodes, the gradient has the correct
sign on each active set, and complementarity holds.

`damage_kkt_metrics` reports constraint and stationarity measures. Compare
these quantities with the classical reference, alongside the field differences.

In [ ]:
from phast.solvers.damage_solver import damage_kkt_metrics


def audit(view, index=-1):
    """KKT metrics for one accepted damage field of a finished run."""
    context = context_at(view, index if index >= 0 else len(view.steps) - 1)
    fem = phast.FEMOperators(view.mesh, material)
    solver = phast.PhaseFieldDamageSolver(fem, tol=1e-10, max_iter=1000)
    damage = torch.as_tensor(view.damage(index), dtype=torch.float64)
    residual = solver.compute_residual(context.history_element, damage)
    return damage_kkt_metrics(residual, damage, context.damage_previous)


rows = []
for name, view in (("classical", reference),
                   ("learned_proposal", proposal),
                   ("learned_replacement", replacement)):
    metrics = audit(view)
    rows.append((name, metrics["kkt_projected_l2"], metrics["kkt_projected_linf"],
                 metrics["kkt_feasible"], metrics["kkt_interior_count"],
                 metrics["kkt_fixed_count"]))

print(f"{'route':22s} {'projected L2':>14s} {'projected Linf':>16s} "
      f"{'feasible':>10s} {'interior':>10s} {'fixed':>8s}")
for name, l2, linf, feasible, interior, fixed in rows:
    print(f"{name:22s} {l2:14.3e} {linf:16.3e} {str(feasible):>10s} "
          f"{interior:10d} {fixed:8d}")

## Key takeaways

- Mechanics remains in PhAST for every damage route.
- A learned initial guess is corrected by the classical damage solve.
- Direct replacement is checked and may use classical fallback; compare complete cost and field accuracy.

### Consolidation

What evidence is needed to assess a direct damage replacement?

<details class="course-hint"><summary>Hint</summary><p>Compare the damage equation, constraints and the reference field.</p></details>

<details class="course-solution"><summary>Conceptual answer</summary><p>Compare projected residuals, damage bounds, irreversibility and field differences with the classical reference. Measure complete runtime, including prediction, checks and fallback.</p></details>

## 14. Exercises

Each exercise asks for a calculation and a short written answer. Record the
wall-clock time and the checkpoint hash of every run you report.

### Exercise 1 — Read the three routes

No new calculation is required.

1. State the final relative L2 difference of each learned route from the
   classical run. Relate the result for `learned_proposal` to its classical
   correction and solver tolerances.
2. Inspect whether the raw prediction lies within $[0,1]$. Explain how a bound
   violation affects acceptance, and which additional checks remain necessary
   when all values are in range.
3. Compare complete runtime on this hardware and identify the main sources of
   additional cost.

**Your answer:**

_Write here._

### Exercise 2 — Loosen the audit

The acceptance thresholds are solver entries, so they can be varied without
touching the model.

1. Rerun `learned_replacement` with `damage_bound_tolerance=0.05` and
   `damage_residual_rtol=1.0e2`, which admits most predictions.
2. Plot the difference trajectory against the classical run. Report where the
   accepted state first departs from the reference and how the departure grows.
3. Audit the final field with `damage_kkt_metrics` and compare its projected
   norms with the classical run.
4. State the conclusion in one sentence: what does a visually plausible crack
   pattern establish about the damage equation?

**Your answer:**

_Write here._

In [ ]:
# Exercise 2: set RUN_THIS to True to execute.
RUN_THIS = False

if RUN_THIS:
    relaxed_dir = run_case(
        "learned_relaxed",
        damage_update="learned_replacement",
        checkpoint=CHECKPOINT,
        damage_bound_tolerance=0.05,
        damage_residual_rtol=1.0e2,
    )
    relaxed = RunViewer(relaxed_dir)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
    reference.plot_damage(-1, ax=axes[0], title="classical")
    relaxed.plot_damage(-1, ax=axes[1], title="learned_replacement, relaxed audit")
    fig.tight_layout()
    plt.show()

    print("final relative L2 difference:",
          f"{relative_l2(relaxed.damage(-1), reference.damage(-1)):.3f} %")
else:
    print("Set RUN_THIS = True to run the relaxed-audit variant.")

### Exercise 3 — Move the specimen away from the training geometry

The network was trained on a different specimen, so this laboratory is already a
transfer test. Explore two additional geometry changes:

1. Rerun the comparison with the hole radius reduced to 0.35 mm, then with the
   three holes moved onto a straight horizontal line at $y = 5$ mm.
2. For each, plot the direct prediction error of Section 10 and report how it
   changes.
3. The boundary indicator this checkpoint was trained with marks the bounding
   box only, so hole boundaries carry no flag. Rerun one case with
   `predictor_options={"boundary_indicator": "topological"}`, which does flag
   them, and report the effect on the prediction error.
4. State what would need to be true of the training set for the transfer to be
   expected to work.

**Your answer:**

_Write here._

### Exercise 4 — Plug in a different architecture

`examples/learned_damage/architectures/template.py` in the checkout is a
commented skeleton with two methods to fill in.

1. Read it and list what a new wrapper must supply that PhAST does not.
2. Write a predictor that returns the previous accepted damage unchanged, run it
   as `learned_replacement`, and explain the audit's verdict.
3. Describe what would change in this notebook if a radial graph neural operator
   replaced the mesh-graph network. Be specific about which cells change and
   which do not.

**Your answer:**

_Write here._

In [ ]:
print((CHECKOUT / "examples/learned_damage/architectures/template.py")
      .read_text()[:1800])

### Exercise 5 — Cost accounting

1. Using `timing_per_step.csv` in each run directory, plot cost per increment
   against increment for the three routes.
2. Identify where the learned routes pay their overhead, and relate it to the
   rejection behaviour you observed.
3. State the conditions under which a learned damage update could reduce
   end-to-end cost, and whether they hold here.

**Your answer:**

_Write here._

## 15. Interpretation and limits

Keep geometry, mesh, material, loading and solver settings fixed when comparing
damage routes. Assess the resulting fields using the classical reference and
projected KKT measures, and report complete timings on the same hardware.

Direct replacement remains experimental. Keep `damage_fallback` enabled in an
initial study and record rejected predictions. Further specimens and
checkpoints are needed to assess how broadly the observed behaviour transfers.

### Further reading

- PhAST documentation: https://cems-lab.github.io/PhAST/
- Learned damage predictor interface: https://cems-lab.github.io/PhAST/user_guide/learned_damage.html
- Miehe, Welschinger and Hofacker (2010), *Thermodynamically consistent phase-field models of fracture*, IJNME 83, 1273–1311.